# Evaluating Learned Embeddings vs Text Embeddings

The v3.0 model gets 17.13% test accuracy on 9-class outcome prediction, but a state-only logistic regression gets **25.47%**. This notebook answers: **are the embeddings themselves good at capturing player identity?**

We compare the learned embeddings (from v3.0 checkpoint) against the LLM-generated text embeddings that served as initialization. The text embeddings are our "prior" — if training improved the space, learned embeddings should:
1. Preserve archetype structure (shooters near shooters)
2. Add basketball-impact information (from actual play-by-play data)
3. Show meaningful career differentiation
4. Maintain or improve cross-season coherence

In [ ]:
# Section 0: Setup & Data Loading
import csv
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score, confusion_matrix as sk_confusion_matrix
from scipy.stats import spearmanr

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# Load checkpoint and extract embeddings
from game_outcome import load_embeddings
from prior_year_init import build_ps_to_base_tensor, build_base_player_mapping

CKPT_PATH = "joint_v21_basecontra.pt"
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=True)
sd = ckpt["state_dict"]

# Learned base embeddings [2310, 384]
learned_base = sd["base_player_emb.weight"].clone()
# Learned composed embeddings [12821, 384] (base + delta)
learned_composed = load_embeddings(CKPT_PATH)
# Text reference embeddings [2310, 384]
text_emb = torch.load("base_player_text_embeddings.pt", map_location="cpu", weights_only=True)

# Load lookup table and name mapping
lu = pd.read_csv("player_season_lookup.csv")
name_map = {}
with open("bbref_name_mapping.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        name_map[row["bbref_id"]] = row["display_name"]

def display_name(bbref_id):
    return name_map.get(bbref_id, bbref_id)

lu["display"] = lu["player"].map(display_name) + " (" + lu["season"].astype(str) + ")"
id_to_name = lu.set_index("player_season_id")["display"].to_dict()
id_to_player = lu.set_index("player_season_id")["player"].to_dict()
id_to_season = lu.set_index("player_season_id")["season"].to_dict()

# ps_to_base mapping
ps_to_base_map, num_base = build_base_player_mapping()
ps_to_base_tensor, _ = build_ps_to_base_tensor(len(lu))

# base_id -> bbref mapping
base_to_player = {}
for _, row in lu.iterrows():
    ps_id = int(row["player_season_id"])
    if ps_id in ps_to_base_map:
        base_id = ps_to_base_map[ps_id]
        if base_id not in base_to_player:
            base_to_player[base_id] = row["player"]

def base_display(base_id):
    bbref = base_to_player.get(base_id, f"ID_{base_id}")
    return display_name(bbref)

# Normalize
text_norm = F.normalize(text_emb, dim=1)
learned_base_norm = F.normalize(learned_base, dim=1)
learned_composed_norm = F.normalize(learned_composed, dim=1)

print(f"Text embeddings:     {text_emb.shape}")
print(f"Learned base:        {learned_base.shape}")
print(f"Learned composed:    {learned_composed.shape}")
print(f"Lookup table:        {len(lu):,} rows")
print(f"Base players:        {num_base:,}")
print(f"Checkpoint:          {CKPT_PATH}")
print(f"Architecture:        {ckpt.get('architecture', 'unknown')}")
print(f"Best epoch:          {ckpt.get('best_epoch', '?')}")
print(f"Best val acc:        {ckpt.get('best_val_acc', 0)*100:.2f}%")

## Section 1: Drift from Initialization

How much did training move each player from their text embedding prior? High-volume players (more training signal) should drift more.

In [ ]:
# Per-player cosine sim between text and learned base embedding
drift_cossim = (text_norm * learned_base_norm).sum(dim=1).numpy()  # [2310]
drift = 1.0 - drift_cossim  # higher = more drifted

# Count possessions per base player
poss_counts_ps = lu.groupby("player_season_id").size()  # would need parquet for real counts
# Approximate: count seasons per base player as proxy for volume
season_counts = lu.groupby("player")["season"].count()
base_season_counts = np.zeros(num_base)
for base_id, bbref in base_to_player.items():
    if base_id < num_base:
        base_season_counts[base_id] = season_counts.get(bbref, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of drift
axes[0].hist(drift_cossim, bins=80, alpha=0.7, color="steelblue", edgecolor="none")
axes[0].axvline(drift_cossim.mean(), color="red", linestyle="--",
                label=f"Mean: {drift_cossim.mean():.4f}")
axes[0].set_xlabel("Cosine Similarity (text vs learned)")
axes[0].set_ylabel("Count")
axes[0].set_title("Drift from Text Initialization")
axes[0].legend()

# Scatter: drift vs player volume (seasons)
axes[1].scatter(base_season_counts, drift_cossim, s=5, alpha=0.3, color="steelblue")
axes[1].set_xlabel("Number of Seasons")
axes[1].set_ylabel("Cosine Similarity (text vs learned)")
axes[1].set_title("Drift vs Player Volume")
# Trend line
z = np.polyfit(base_season_counts, drift_cossim, 1)
x_line = np.linspace(base_season_counts.min(), base_season_counts.max(), 100)
axes[1].plot(x_line, np.polyval(z, x_line), color="red", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

# Top 10 most-drifted and least-drifted
print("\nTop 10 MOST drifted (lowest cosine sim text vs learned):")
order = np.argsort(drift_cossim)
for idx in order[:10]:
    print(f"  {base_display(idx):30s} sim={drift_cossim[idx]:.4f} seasons={int(base_season_counts[idx])}")

print("\nTop 10 LEAST drifted (highest cosine sim):")
for idx in order[-10:]:
    print(f"  {base_display(idx):30s} sim={drift_cossim[idx]:.4f} seasons={int(base_season_counts[idx])}")

print(f"\nOverall: mean sim={drift_cossim.mean():.4f}, std={drift_cossim.std():.4f}")
corr = np.corrcoef(base_season_counts, drift_cossim)[0, 1]
print(f"Correlation (seasons vs sim): {corr:.4f} (negative = high-volume players drift more)")

## Section 2: Nearest-Neighbor Comparison

For notable players, show side-by-side text NN vs learned NN. Then measure NN overlap across all players.

In [ ]:
def find_nn_base(query_bbref, emb_norm_mat, k=8):
    """Find k nearest base-player neighbors."""
    base_id = None
    for bid, p in base_to_player.items():
        if p == query_bbref:
            base_id = bid
            break
    if base_id is None:
        return None
    sims = (emb_norm_mat @ emb_norm_mat[base_id]).numpy()
    top_ids = np.argsort(-sims)
    results = []
    for idx in top_ids:
        if idx == base_id:
            continue
        results.append((base_display(idx), sims[idx]))
        if len(results) >= k:
            break
    return results

notable = [
    ("jamesle01", "LeBron James"),
    ("curryst01", "Stephen Curry"),
    ("jokicni01", "Nikola Jokic"),
    ("greendr01", "Draymond Green"),
    ("redicjj01", "J.J. Redick"),
    ("rodmade01", "Dennis Rodman"),
    ("duranke01", "Kevin Durant"),
    ("howardw01", "Dwight Howard"),
]

for bbref, pname in notable:
    text_nn = find_nn_base(bbref, text_norm, k=8)
    learned_nn = find_nn_base(bbref, learned_base_norm, k=8)
    if text_nn is None:
        continue
    print(f"\n{'='*70}")
    print(f"  {pname}")
    print(f"  {'Text NN':40s} | {'Learned NN':40s}")
    print(f"  {'-'*40} | {'-'*40}")
    for i in range(8):
        t_name, t_sim = text_nn[i]
        l_name, l_sim = learned_nn[i]
        print(f"  {i+1}. {t_name:30s} {t_sim:.3f} | {i+1}. {l_name:30s} {l_sim:.3f}")

In [ ]:
# NN overlap (Jaccard) for all base players at K=5, 10, 25
def nn_overlap(emb_a, emb_b, k):
    """Compute per-player Jaccard overlap of top-K neighbors between two spaces."""
    sim_a = emb_a @ emb_a.T
    sim_b = emb_b @ emb_b.T
    n = emb_a.shape[0]
    jaccards = []
    for i in range(n):
        nn_a = set(sim_a[i].topk(k + 1).indices.tolist()) - {i}
        nn_b = set(sim_b[i].topk(k + 1).indices.tolist()) - {i}
        # Take first k after removing self
        nn_a = set(list(nn_a)[:k])
        nn_b = set(list(nn_b)[:k])
        j = len(nn_a & nn_b) / len(nn_a | nn_b) if len(nn_a | nn_b) > 0 else 0
        jaccards.append(j)
    return np.array(jaccards)

for k in [5, 10, 25]:
    j = nn_overlap(text_norm, learned_base_norm, k)
    print(f"NN@{k:2d} Jaccard: mean={j.mean():.4f}, median={np.median(j):.4f}")

# Histogram of NN@10 Jaccard
j10 = nn_overlap(text_norm, learned_base_norm, 10)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(j10, bins=50, alpha=0.7, color="steelblue", edgecolor="none")
ax.axvline(j10.mean(), color="red", linestyle="--", label=f"Mean: {j10.mean():.3f}")
ax.set_xlabel("Jaccard Overlap (NN@10)")
ax.set_ylabel("Count")
ax.set_title("Per-Player NN@10 Overlap: Text vs Learned")
ax.legend()
plt.tight_layout()
plt.show()

## Section 3: Global Structure - Rank Correlation

Spearman correlation between all pairwise similarities in text vs learned space. Shows whether the global ordering of "who is similar to whom" was preserved.

In [ ]:
# Full pairwise similarity matrices
text_sim_full = (text_norm @ text_norm.T).numpy()
learned_sim_full = (learned_base_norm @ learned_base_norm.T).numpy()

# Upper triangle (exclude diagonal)
n = text_sim_full.shape[0]
triu_idx = np.triu_indices(n, k=1)
text_pairs = text_sim_full[triu_idx]
learned_pairs = learned_sim_full[triu_idx]

# Spearman rank correlation
rho, pval = spearmanr(text_pairs, learned_pairs)
print(f"Spearman rho: {rho:.4f} (p={pval:.2e})")
print(f"Total pairs: {len(text_pairs):,}")

# Scatter plot (random sample for visibility)
np.random.seed(42)
n_sample = 20000
sample_idx = np.random.choice(len(text_pairs), n_sample, replace=False)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(text_pairs[sample_idx], learned_pairs[sample_idx],
           s=2, alpha=0.15, color="steelblue")
ax.plot([0, 1], [0, 1], "r--", alpha=0.5, label="y=x")
ax.set_xlabel("Text Pairwise Similarity")
ax.set_ylabel("Learned Pairwise Similarity")
ax.set_title(f"Global Structure Preservation (Spearman rho={rho:.3f})")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## Section 4: Same-Player Cross-Season Coherence

For each player with 2+ seasons, compute mean cosine sim across their seasons in the learned composed space. Compare to text embeddings.

In [ ]:
# Load per-season text embeddings for cross-season comparison
text_ps_emb = torch.load("player_text_embeddings.pt", map_location="cpu", weights_only=True)
text_ps_norm = F.normalize(text_ps_emb, dim=1)

multi_season = lu.groupby("player").filter(lambda x: len(x) >= 2)
players_multi = multi_season["player"].unique()

text_coherence = []
learned_coherence = []
player_names_coherence = []

for player in players_multi:
    rows = lu[lu["player"] == player].sort_values("season")
    pids = rows["player_season_id"].values
    if len(pids) < 2:
        continue

    # Text cross-season sim (per-season embeddings)
    text_sims = []
    for i in range(len(pids)):
        for j in range(i + 1, len(pids)):
            text_sims.append((text_ps_norm[pids[i]] @ text_ps_norm[pids[j]]).item())

    # Learned cross-season sim (composed embeddings)
    learned_sims = []
    for i in range(len(pids)):
        for j in range(i + 1, len(pids)):
            learned_sims.append((learned_composed_norm[pids[i]] @ learned_composed_norm[pids[j]]).item())

    text_coherence.append(np.mean(text_sims))
    learned_coherence.append(np.mean(learned_sims))
    player_names_coherence.append(display_name(player))

text_coherence = np.array(text_coherence)
learned_coherence = np.array(learned_coherence)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Side-by-side histograms
axes[0].hist(text_coherence, bins=60, alpha=0.6, label=f"Text (mean={text_coherence.mean():.3f})",
             color="coral", density=True)
axes[0].hist(learned_coherence, bins=60, alpha=0.6, label=f"Learned (mean={learned_coherence.mean():.3f})",
             color="steelblue", density=True)
axes[0].set_xlabel("Mean Cross-Season Cosine Similarity")
axes[0].set_ylabel("Density")
axes[0].set_title("Cross-Season Coherence Distribution")
axes[0].legend()

# Scatter: text vs learned coherence
axes[1].scatter(text_coherence, learned_coherence, s=5, alpha=0.3, color="steelblue")
axes[1].plot([0, 1], [0, 1], "r--", alpha=0.5, label="y=x")
axes[1].set_xlabel("Text Coherence")
axes[1].set_ylabel("Learned Coherence")
axes[1].set_title("Cross-Season Coherence: Text vs Learned")
axes[1].legend()
axes[1].set_aspect("equal")

plt.tight_layout()
plt.show()

# What fraction are below diagonal (training hurt coherence)?
below_diag = (learned_coherence < text_coherence).mean()
print(f"Players where training REDUCED coherence: {below_diag*100:.1f}%")
print(f"Text coherence:    mean={text_coherence.mean():.4f}, median={np.median(text_coherence):.4f}")
print(f"Learned coherence: mean={learned_coherence.mean():.4f}, median={np.median(learned_coherence):.4f}")

## Section 5: Career Trajectory Comparison

Side-by-side line plots: year-to-year similarity and drift from rookie season in text vs learned space.

In [ ]:
evolution_players = [
    "jamesle01", "curryst01", "lopezbr01", "cartevi01",
    "howardw01", "griffbl01", "westbru01", "paulch01",
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for bbref in evolution_players:
    rows = lu[lu["player"] == bbref].sort_values("season")
    if len(rows) < 3:
        continue
    dname = display_name(bbref)
    pids = rows["player_season_id"].values
    seasons = rows["season"].values

    # Year-to-year: text
    text_yty = [(text_ps_norm[pids[i]] @ text_ps_norm[pids[i+1]]).item() for i in range(len(pids)-1)]
    # Year-to-year: learned
    learned_yty = [(learned_composed_norm[pids[i]] @ learned_composed_norm[pids[i+1]]).item()
                   for i in range(len(pids)-1)]

    axes[0, 0].plot(seasons[1:], text_yty, marker="o", markersize=3, label=dname[:20], alpha=0.8)
    axes[0, 1].plot(seasons[1:], learned_yty, marker="o", markersize=3, label=dname[:20], alpha=0.8)

    # Drift from rookie: text
    text_drift = [(text_ps_norm[pids[0]] @ text_ps_norm[pid]).item() for pid in pids]
    # Drift from rookie: learned
    learned_drift = [(learned_composed_norm[pids[0]] @ learned_composed_norm[pid]).item() for pid in pids]

    axes[1, 0].plot(seasons, text_drift, marker="o", markersize=3, label=dname[:20], alpha=0.8)
    axes[1, 1].plot(seasons, learned_drift, marker="o", markersize=3, label=dname[:20], alpha=0.8)

axes[0, 0].set_title("Text: Year-to-Year Similarity")
axes[0, 0].set_ylabel("Cosine Sim (consecutive)")
axes[0, 0].set_ylim(0.5, 1.0)
axes[0, 0].legend(fontsize=7, ncol=2, loc="lower left")

axes[0, 1].set_title("Learned: Year-to-Year Similarity")
axes[0, 1].set_ylabel("Cosine Sim (consecutive)")
axes[0, 1].set_ylim(0.5, 1.0)
axes[0, 1].legend(fontsize=7, ncol=2, loc="lower left")

axes[1, 0].set_title("Text: Drift from Rookie Season")
axes[1, 0].set_ylabel("Cosine Sim (vs first season)")
axes[1, 0].set_xlabel("Season")
axes[1, 0].set_ylim(0.3, 1.0)
axes[1, 0].legend(fontsize=7, ncol=2, loc="lower left")

axes[1, 1].set_title("Learned: Drift from Rookie Season")
axes[1, 1].set_ylabel("Cosine Sim (vs first season)")
axes[1, 1].set_xlabel("Season")
axes[1, 1].set_ylim(0.3, 1.0)
axes[1, 1].legend(fontsize=7, ncol=2, loc="lower left")

plt.suptitle("Career Trajectories: Text vs Learned Embeddings", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Section 6: Delta Analysis

Distribution of delta norms (the season-specific component). Which player-seasons diverged most from their base?

In [ ]:
# Extract deltas from checkpoint
delta_raw = sd["delta_raw.weight"]  # [12821, 64]
delta_proj_w = sd["delta_proj.weight"]  # [384, 64]
delta_projected = delta_raw @ delta_proj_w.T  # [12821, 384]
delta_norms = delta_projected.norm(dim=1).numpy()

base_norms_all = learned_base[ps_to_base_tensor].norm(dim=1).numpy()  # [12821]
delta_ratio = delta_norms / (base_norms_all + 1e-8)

# Season counts per player-season (as proxy for volume)
season_counts_ps = np.array([id_to_season.get(i, 2010) for i in range(len(lu))])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution of delta norms
axes[0].hist(delta_norms, bins=80, alpha=0.7, color="steelblue", edgecolor="none")
axes[0].axvline(delta_norms.mean(), color="red", linestyle="--",
                label=f"Mean: {delta_norms.mean():.4f}")
axes[0].set_xlabel("Delta Norm (projected, 384-dim)")
axes[0].set_ylabel("Count")
axes[0].set_title("Delta Norm Distribution")
axes[0].legend()

# Delta ratio distribution
axes[1].hist(delta_ratio, bins=80, alpha=0.7, color="coral", edgecolor="none")
axes[1].axvline(delta_ratio.mean(), color="red", linestyle="--",
                label=f"Mean: {delta_ratio.mean():.2%}")
axes[1].set_xlabel("|delta| / |base|")
axes[1].set_ylabel("Count")
axes[1].set_title("Delta/Base Ratio")
axes[1].legend()

# Scatter: delta norm vs number of seasons for base player
ps_base_season_counts = np.array([base_season_counts[ps_to_base_tensor[i].item()]
                                   if ps_to_base_tensor[i].item() < len(base_season_counts)
                                   else 1 for i in range(len(lu))])
axes[2].scatter(ps_base_season_counts, delta_norms, s=3, alpha=0.1, color="steelblue")
axes[2].set_xlabel("Player's Total Seasons")
axes[2].set_ylabel("Delta Norm")
axes[2].set_title("Delta Norm vs Player Longevity")

plt.tight_layout()
plt.show()

# Top 20 largest deltas
print("Top 20 largest deltas:")
top_delta_idx = np.argsort(-delta_norms)[:20]
for rank, idx in enumerate(top_delta_idx, 1):
    name = id_to_name.get(idx, f"ID_{idx}")
    print(f"  {rank:2d}. {name:35s} delta={delta_norms[idx]:.4f} ratio={delta_ratio[idx]:.2%}")

print(f"\nDelta stats: mean={delta_norms.mean():.4f}, max={delta_norms.max():.4f}")
print(f"Delta/base ratio: mean={delta_ratio.mean():.2%}, max={delta_ratio.max():.2%}")

## Section 7: Effective Dimensionality (SVD Spectrum)

SVD on text, learned base, and learned composed embeddings. Fewer effective dims = information compressed/lost; more = training added structure.

In [ ]:
def svd_spectrum(emb, label, n_sample=3000):
    """Compute SVD spectrum for an embedding matrix."""
    idx = np.random.choice(emb.shape[0], min(n_sample, emb.shape[0]), replace=False)
    sample = F.normalize(emb[idx], dim=1)
    U, S, V = torch.svd(sample)
    explained = (S ** 2) / (S ** 2).sum()
    cumulative = explained.cumsum(0).numpy()
    n_90 = np.searchsorted(cumulative, 0.9) + 1
    return cumulative, n_90

np.random.seed(42)
text_cum, text_n90 = svd_spectrum(text_emb, "Text")
base_cum, base_n90 = svd_spectrum(learned_base, "Learned Base")
composed_cum, composed_n90 = svd_spectrum(learned_composed, "Learned Composed")

fig, ax = plt.subplots(figsize=(10, 6))
n_dims = 80
ax.plot(range(1, n_dims+1), text_cum[:n_dims], label=f"Text (90% at {text_n90} dims)",
        marker="o", markersize=3)
ax.plot(range(1, n_dims+1), base_cum[:n_dims], label=f"Learned Base (90% at {base_n90} dims)",
        marker="s", markersize=3)
ax.plot(range(1, n_dims+1), composed_cum[:n_dims], label=f"Learned Composed (90% at {composed_n90} dims)",
        marker="^", markersize=3)
ax.axhline(0.9, color="red", linestyle="--", alpha=0.3)
ax.set_xlabel("Number of Components")
ax.set_ylabel("Cumulative Variance Explained")
ax.set_title("SVD Spectrum: Effective Dimensionality")
ax.legend()
plt.tight_layout()
plt.show()

print(f"90% variance threshold:")
print(f"  Text:             {text_n90} dims")
print(f"  Learned Base:     {base_n90} dims")
print(f"  Learned Composed: {composed_n90} dims")

## Section 8: Archetype Preservation

K-means on text defines "ground truth" archetypes. K-means on learned base shows if training preserved, merged, or split roles.

In [ ]:
n_clusters = 8

km_text = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
text_labels = km_text.fit_predict(text_norm.numpy())

km_learned = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
learned_labels = km_learned.fit_predict(learned_base_norm.numpy())

# NMI between text and learned clusters
nmi = normalized_mutual_info_score(text_labels, learned_labels)
print(f"NMI (text vs learned clusters): {nmi:.4f}")
print(f"(1.0 = identical clustering, 0.0 = independent)")

# Cluster transfer accuracy: assign learned embeddings to nearest text centroid
text_centroids = torch.from_numpy(km_text.cluster_centers_).float()
text_centroids_norm = F.normalize(text_centroids, dim=1)
transfer_labels = (learned_base_norm @ text_centroids_norm.T).argmax(dim=1).numpy()
transfer_acc = (transfer_labels == text_labels).mean()
print(f"Cluster transfer accuracy: {transfer_acc*100:.1f}%")

# Confusion matrix
cm = sk_confusion_matrix(text_labels, learned_labels)
# Normalize by row (text cluster)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", ax=ax,
            xticklabels=[f"L{i}" for i in range(n_clusters)],
            yticklabels=[f"T{i}" for i in range(n_clusters)])
ax.set_xlabel("Learned Cluster")
ax.set_ylabel("Text Cluster")
ax.set_title(f"Archetype Preservation (NMI={nmi:.3f})")
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side t-SNE: text vs learned, colored by respective clusters
np.random.seed(42)
n_tsne = min(2500, num_base)
tsne_idx = np.random.choice(num_base, n_tsne, replace=False)

# Add notable players
notable_base_ids = []
for bbref, _ in notable:
    for bid, p in base_to_player.items():
        if p == bbref:
            notable_base_ids.append(bid)
            break
tsne_idx = np.unique(np.concatenate([tsne_idx, notable_base_ids]))
n_tsne = len(tsne_idx)

print(f"Running t-SNE on {n_tsne} base players (text space)...")
coords_text = TSNE(n_components=2, perplexity=40, random_state=42,
                   max_iter=1000, init="pca").fit_transform(text_norm[tsne_idx].numpy())

print(f"Running t-SNE on {n_tsne} base players (learned space)...")
coords_learned = TSNE(n_components=2, perplexity=40, random_state=42,
                      max_iter=1000, init="pca").fit_transform(learned_base_norm[tsne_idx].numpy())

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
palette = sns.color_palette("husl", n_clusters)

for c in range(n_clusters):
    mask = text_labels[tsne_idx] == c
    axes[0].scatter(coords_text[mask, 0], coords_text[mask, 1],
                    color=palette[c], s=8, alpha=0.5, label=f"T{c}")

for c in range(n_clusters):
    mask = learned_labels[tsne_idx] == c
    axes[1].scatter(coords_learned[mask, 0], coords_learned[mask, 1],
                    color=palette[c], s=8, alpha=0.5, label=f"L{c}")

# Annotate notable players
for bbref, pname in notable:
    for bid, p in base_to_player.items():
        if p == bbref:
            loc = np.where(tsne_idx == bid)[0]
            if len(loc) > 0:
                idx = loc[0]
                short = pname.split()[-1]
                for ax_i in [0, 1]:
                    c = [coords_text, coords_learned][ax_i]
                    axes[ax_i].annotate(short, (c[idx, 0], c[idx, 1]),
                                       fontsize=7, fontweight="bold",
                                       bbox=dict(boxstyle="round,pad=0.1",
                                                 facecolor="white", alpha=0.7))
            break

axes[0].set_title("Text Embeddings (colored by text clusters)")
axes[0].legend(fontsize=7, markerscale=2, loc="best")
axes[1].set_title("Learned Embeddings (colored by learned clusters)")
axes[1].legend(fontsize=7, markerscale=2, loc="best")

plt.suptitle("t-SNE: Text vs Learned Base Embeddings", fontsize=14)
plt.tight_layout()
plt.show()

## Section 9: Cross-Era Preservation

Do cross-era archetype pairs (Nash/CP3, Carter/Edwards, etc.) maintain their similarity after training?

In [ ]:
cross_era_pairs = [
    ("nashst01", 2005, "paulch01", 2008, "Floor Generals"),
    ("cartevi01", 2001, "edwaran01", 2022, "Athletic Wings"),
    ("nowitdi01", 2006, "townska01", 2019, "Stretch Bigs"),
    ("wallabe01", 2004, "greendr01", 2016, "Defensive Anchors"),
    ("iversal01", 2001, "westbru01", 2017, "Volume Scorers"),
    ("diawbo01", 2014, "jokicni01", 2023, "Playmaking Bigs"),
]

labels_ce, text_sims_ce, learned_sims_ce = [], [], []

for p1, s1, p2, s2, label in cross_era_pairs:
    m1 = lu[(lu["player"] == p1) & (lu["season"] == s1)]
    m2 = lu[(lu["player"] == p2) & (lu["season"] == s2)]
    if len(m1) == 0 or len(m2) == 0:
        continue
    id1, id2 = m1.iloc[0]["player_season_id"], m2.iloc[0]["player_season_id"]

    t_sim = (text_ps_norm[id1] @ text_ps_norm[id2]).item()
    l_sim = (learned_composed_norm[id1] @ learned_composed_norm[id2]).item()

    labels_ce.append(label)
    text_sims_ce.append(t_sim)
    learned_sims_ce.append(l_sim)

    n1 = f"{display_name(p1)} ({s1})"
    n2 = f"{display_name(p2)} ({s2})"
    delta = l_sim - t_sim
    print(f"  {label:20s}: text={t_sim:.4f}  learned={l_sim:.4f}  delta={delta:+.4f}")

# Bar chart comparison
x = np.arange(len(labels_ce))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, text_sims_ce, width, label="Text", color="coral", alpha=0.8)
ax.bar(x + width/2, learned_sims_ce, width, label="Learned", color="steelblue", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels_ce, rotation=15, ha="right")
ax.set_ylabel("Cosine Similarity")
ax.set_title("Cross-Era Archetype Similarity: Text vs Learned")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## Section 10: Downstream Game Prediction

Logistic regression for game win prediction using 3 embedding types: text, learned base, and learned composed.

In [ ]:
from game_outcome import build_game_table, games_to_embedding_features, evaluate_model

games = build_game_table("possessions.parquet")
train = games[games["season"] < 2019]
test = games[games["season"] >= 2021]
y_train = train["home_win"].values
y_test = test["home_win"].values
home_rate = y_test.mean()

print(f"Games: {len(train):,} train / {len(test):,} test")
print(f"Home win rate (test): {home_rate:.1%}")

# (a) Text embeddings expanded to player-season level
text_ps = text_emb[ps_to_base_tensor]  # [12821, 384]
X_train_text = games_to_embedding_features(train, text_ps)
X_test_text = games_to_embedding_features(test, text_ps)
acc_text, _ = evaluate_model("Text", X_train_text, y_train, X_test_text, y_test)
print(f"Text embeddings:       {acc_text*100:.2f}%")

# (b) Learned base embeddings expanded
learned_base_ps = learned_base[ps_to_base_tensor]  # [12821, 384]
X_train_base = games_to_embedding_features(train, learned_base_ps)
X_test_base = games_to_embedding_features(test, learned_base_ps)
acc_base, _ = evaluate_model("Learned Base", X_train_base, y_train, X_test_base, y_test)
print(f"Learned base:          {acc_base*100:.2f}%")

# (c) Learned composed embeddings (full with deltas)
X_train_composed = games_to_embedding_features(train, learned_composed)
X_test_composed = games_to_embedding_features(test, learned_composed)
acc_composed, _ = evaluate_model("Learned Composed", X_train_composed, y_train, X_test_composed, y_test)
print(f"Learned composed:      {acc_composed*100:.2f}%")

# Bar chart
labels_gp = ["Home Always", "Text Emb", "Learned Base", "Learned Composed"]
accs = [home_rate, acc_text, acc_base, acc_composed]
colors = ["gray", "coral", "steelblue", "darkblue"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels_gp, [a * 100 for a in accs], color=colors, alpha=0.8)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{acc*100:.1f}%", ha="center", fontsize=10)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Downstream Game Prediction")
ax.set_ylim(40, 70)
plt.tight_layout()
plt.show()

print(f"\nDelta composed vs text: {(acc_composed - acc_text)*100:+.2f}pp")
print(f"Delta composed vs base: {(acc_composed - acc_base)*100:+.2f}pp")

## Section 11: Summary Dashboard

All metrics side by side.

In [ ]:
print("=" * 70)
print("EMBEDDING QUALITY SUMMARY")
print("=" * 70)
print(f"")
print(f"{'Metric':<40s} {'Value':>15s}")
print(f"{'-'*40} {'-'*15}")

print(f"{'Drift from init (mean cosine sim)':<40s} {drift_cossim.mean():>15.4f}")
print(f"{'Drift-volume correlation':<40s} {corr:>15.4f}")
print(f"{'NN@10 Jaccard (text vs learned)':<40s} {j10.mean():>15.4f}")
print(f"{'Global structure Spearman rho':<40s} {rho:>15.4f}")
print(f"{'Cross-season coherence (text)':<40s} {text_coherence.mean():>15.4f}")
print(f"{'Cross-season coherence (learned)':<40s} {learned_coherence.mean():>15.4f}")
print(f"{'Coherence reduced (%)':<40s} {below_diag*100:>14.1f}%")
print(f"{'SVD 90% dims (text)':<40s} {text_n90:>15d}")
print(f"{'SVD 90% dims (learned base)':<40s} {base_n90:>15d}")
print(f"{'SVD 90% dims (learned composed)':<40s} {composed_n90:>15d}")
print(f"{'Archetype NMI':<40s} {nmi:>15.4f}")
print(f"{'Cluster transfer accuracy':<40s} {transfer_acc*100:>14.1f}%")
print(f"{'Delta norm mean':<40s} {delta_norms.mean():>15.4f}")
print(f"{'Delta/base ratio mean':<40s} {delta_ratio.mean()*100:>14.2f}%")
print(f"{'Game prediction (text)':<40s} {acc_text*100:>14.2f}%")
print(f"{'Game prediction (learned base)':<40s} {acc_base*100:>14.2f}%")
print(f"{'Game prediction (learned composed)':<40s} {acc_composed*100:>14.2f}%")
print(f"{'Game prediction (home always)':<40s} {home_rate*100:>14.2f}%")

print(f"\n{'='*70}")
print("QUALITATIVE VERDICT:")
if rho > 0.8 and nmi > 0.5:
    print("  Training PRESERVED the embedding space structure well.")
elif rho > 0.5:
    print("  Training PARTIALLY preserved structure — some reorganization occurred.")
else:
    print("  Training SIGNIFICANTLY reorganized the embedding space.")

if acc_composed > acc_text:
    print(f"  Composed embeddings IMPROVED downstream prediction (+{(acc_composed-acc_text)*100:.1f}pp over text).")
else:
    print(f"  Composed embeddings DID NOT improve downstream prediction ({(acc_composed-acc_text)*100:+.1f}pp vs text).")

if learned_coherence.mean() > text_coherence.mean() * 0.95:
    print("  Cross-season coherence was MAINTAINED.")
else:
    print("  Cross-season coherence DEGRADED during training.")
print("=" * 70)